In [1]:
%pip install -q scikit-learn

In [2]:
from pathlib import Path

DATA_DIR = Path("sample_documents")
DATA_DIR.mkdir(exist_ok=True)

documents = {
    "codenova.txt": """
CodeNova Internship Program

CodeNova provides hands-on internships in Artificial Intelligence,
Machine Learning, and Web Development.

Interns complete practical projects to develop technical skills.
Python is commonly used for data analysis and machine learning tasks.

Internship projects help students practise problem-solving,
software development, and project documentation.
""",
    "machine_learning.txt": """
Machine Learning Basics

Machine Learning is a branch of Artificial Intelligence.
It enables computer systems to learn patterns from data.

Supervised learning uses labelled training data.
Classification predicts categories, while regression predicts
continuous numerical values.

Model performance can be evaluated using suitable metrics.
For regression, common metrics include MAE, MSE, and R-squared.
""",
    "rag_assistant.txt": """
Question Answering and RAG

A Question-Answering Assistant responds to questions using
information available in a collection of documents.

Retrieval-Augmented Generation, or RAG, retrieves relevant
document passages before generating or constructing an answer.

TF-IDF represents text using term importance.
Cosine similarity compares the direction of numerical vectors.

Source citations help users verify answers against the documents.
If the documents do not support an answer, the assistant should
clearly state that the information was not found.
"""
}

for filename, content in documents.items():
    file_path = DATA_DIR / filename
    file_path.write_text(content.strip(), encoding="utf-8")

print("Sample documents created successfully:")
for file_path in sorted(DATA_DIR.glob("*.txt")):
    print("-", file_path.name)

Sample documents created successfully:
- codenova.txt
- machine_learning.txt
- rag_assistant.txt


In [3]:
import re

def load_and_chunk_documents(folder, max_words=100):
    chunks = []

    for file_path in sorted(Path(folder).glob("*.txt")):
        text = file_path.read_text(encoding="utf-8")

        # Split at paragraph boundaries first
        paragraphs = [
            p.strip()
            for p in re.split(r"\n\s*\n", text)
            if p.strip()
        ]

        for paragraph in paragraphs:
            # Split long paragraphs at sentence boundaries
            sentences = re.split(r"(?<=[.!?])\s+", paragraph)

            current_chunk = []

            for sentence in sentences:
                if not sentence.strip():
                    continue

                proposed = " ".join(current_chunk + [sentence])

                if current_chunk and len(proposed.split()) > max_words:
                    chunks.append({
                        "source": file_path.name,
                        "text": " ".join(current_chunk)
                    })
                    current_chunk = [sentence]
                else:
                    current_chunk.append(sentence)

            if current_chunk:
                chunks.append({
                    "source": file_path.name,
                    "text": " ".join(current_chunk)
                })

    return chunks


chunks = load_and_chunk_documents(DATA_DIR)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i} | Source: {chunk['source']}")
    print(chunk["text"])

Total chunks: 13

Chunk 1 | Source: codenova.txt
CodeNova Internship Program

Chunk 2 | Source: codenova.txt
CodeNova provides hands-on internships in Artificial Intelligence,
Machine Learning, and Web Development.

Chunk 3 | Source: codenova.txt
Interns complete practical projects to develop technical skills. Python is commonly used for data analysis and machine learning tasks.

Chunk 4 | Source: codenova.txt
Internship projects help students practise problem-solving,
software development, and project documentation.

Chunk 5 | Source: machine_learning.txt
Machine Learning Basics

Chunk 6 | Source: machine_learning.txt
Machine Learning is a branch of Artificial Intelligence. It enables computer systems to learn patterns from data.

Chunk 7 | Source: machine_learning.txt
Supervised learning uses labelled training data. Classification predicts categories, while regression predicts
continuous numerical values.

Chunk 8 | Source: machine_learning.txt
Model performance can be evaluated usin

In [4]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class LocalQAAssistant:
    def __init__(self, chunks, threshold=0.15, top_k=3):
        if not chunks:
            raise ValueError("No document chunks were provided.")

        self.chunks = chunks
        self.threshold = threshold
        self.top_k = top_k

        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            sublinear_tf=True
        )

        self.document_vectors = self.vectorizer.fit_transform(
            [chunk["text"] for chunk in chunks]
        )

    def retrieve(self, query):
        query = query.strip()

        if not query:
            return []

        query_vector = self.vectorizer.transform([query])

        scores = cosine_similarity(
            query_vector,
            self.document_vectors
        ).flatten()

        ranked_indices = np.argsort(scores)[::-1][:self.top_k]

        results = []
        for index in ranked_indices:
            results.append({
                "source": self.chunks[index]["source"],
                "passage": self.chunks[index]["text"],
                "score": float(scores[index])
            })

        return results

    def answer_query(self, query):
        results = self.retrieve(query)

        if not query.strip():
            return {
                "answer": "Please enter a question.",
                "source_passage": None,
                "source": None,
                "score": 0.0,
                "retrieved_passages": []
            }

        if not results or results[0]["score"] < self.threshold:
            return {
                "answer": "Information not found in provided document",
                "source_passage": None,
                "source": None,
                "score": results[0]["score"] if results else 0.0,
                "retrieved_passages": results
            }

        best = results[0]

        return {
            # Extractive answer: use the actual document passage.
            "answer": best["passage"],
            "source_passage": best["passage"],
            "source": best["source"],
            "score": best["score"],
            "retrieved_passages": results
        }


assistant = LocalQAAssistant(chunks)

print("Local QA Assistant initialized successfully.")
print("Indexed passages:", len(chunks))
print("Retrieval method: TF-IDF + Cosine Similarity")

Local QA Assistant initialized successfully.
Indexed passages: 13
Retrieval method: TF-IDF + Cosine Similarity


In [5]:
def display_answer(query):
    result = assistant.answer_query(query)

    print("=" * 65)
    print("QUESTION:", query)
    print("\nANSWER:")
    print(result["answer"])

    if result["source_passage"] is not None:
        print("\nSOURCE DOCUMENT:", result["source"])
        print("SOURCE PASSAGE:")
        print(result["source_passage"])
        print(f"\nRETRIEVAL SCORE: {result['score']:.4f}")
    else:
        print("\nSOURCE DOCUMENT: None")
        print("No supporting source passage was accepted.")

    print("\nTOP RETRIEVED PASSAGES:")
    for i, passage in enumerate(result["retrieved_passages"], start=1):
        print(
            f"{i}. {passage['source']} "
            f"(score: {passage['score']:.4f})"
        )
        print(passage["passage"])

    return result


supported_result = display_answer(
    "What does CodeNova provide?"
)

QUESTION: What does CodeNova provide?

ANSWER:
CodeNova Internship Program

SOURCE DOCUMENT: codenova.txt
SOURCE PASSAGE:
CodeNova Internship Program

RETRIEVAL SCORE: 0.4071

TOP RETRIEVED PASSAGES:
1. codenova.txt (score: 0.4071)
CodeNova Internship Program
2. codenova.txt (score: 0.2152)
CodeNova provides hands-on internships in Artificial Intelligence,
Machine Learning, and Web Development.
3. rag_assistant.txt (score: 0.0000)
Retrieval-Augmented Generation, or RAG, retrieves relevant
document passages before generating or constructing an answer.


In [6]:
unsupported_result = display_answer(
    "What is the capital of France?"
)

assert (
    unsupported_result["answer"]
    == "Information not found in provided document"
), "Unsupported query was not rejected. Review the threshold and retrieval results."

print("\nUnsupported-query test passed.")

QUESTION: What is the capital of France?

ANSWER:
Information not found in provided document

SOURCE DOCUMENT: None
No supporting source passage was accepted.

TOP RETRIEVED PASSAGES:
1. rag_assistant.txt (score: 0.0000)
Source citations help users verify answers against the documents. If the documents do not support an answer, the assistant should
clearly state that the information was not found.
2. rag_assistant.txt (score: 0.0000)
TF-IDF represents text using term importance. Cosine similarity compares the direction of numerical vectors.
3. rag_assistant.txt (score: 0.0000)
Retrieval-Augmented Generation, or RAG, retrieves relevant
document passages before generating or constructing an answer.

Unsupported-query test passed.


In [7]:
test_cases = [
    {
        "question": "What does CodeNova provide?",
        "supported": True
    },
    {
        "question": "What is supervised learning?",
        "supported": True
    },
    {
        "question": "What is TF-IDF?",
        "supported": True
    },
    {
        "question": "What is the capital of France?",
        "supported": False
    },
    {
        "question": "Who won the FIFA World Cup in 1998?",
        "supported": False
    }
]

test_results = []

for case in test_cases:
    result = assistant.answer_query(case["question"])

    predicted_supported = result["source_passage"] is not None
    correct = predicted_supported == case["supported"]

    test_results.append({
        "Question": case["question"],
        "Expected": (
            "Supported" if case["supported"] else "Unsupported"
        ),
        "Predicted": (
            "Supported" if predicted_supported else "Unsupported"
        ),
        "Correct": correct,
        "Score": round(result["score"], 4)
    })

import pandas as pd

results_df = pd.DataFrame(test_results)
display(results_df)

total = len(test_results)
correct_count = sum(row["Correct"] for row in test_results)
accuracy = correct_count / total * 100

print(f"Correct support decisions: {correct_count}/{total}")
print(f"Support-detection accuracy: {accuracy:.2f}%")

,Question,Expected,Predicted,Correct,Score
0,What does CodeNova provide?,Supported,Supported,True,0.4071
1,What is supervised learning?,Supported,Supported,True,0.3015
2,What is TF-IDF?,Supported,Supported,True,0.3512
3,What is the capital of France?,Unsupported,Unsupported,True,0.0000
4,Who won the FIFA World Cup in 1998?,Unsupported,Unsupported,True,0.0000


Correct support decisions: 5/5
Support-detection accuracy: 100.00%


In [8]:
while True:
    question = input(
        "\nEnter your question (or type 'exit' to stop): "
    ).strip()

    if question.lower() == "exit":
        print("Assistant closed.")
        break

    if not question:
        print("Please enter a question.")
        continue

    display_answer(question)


Enter your question (or type 'exit' to stop): What does CodeNova provide?
QUESTION: What does CodeNova provide?

ANSWER:
CodeNova Internship Program

SOURCE DOCUMENT: codenova.txt
SOURCE PASSAGE:
CodeNova Internship Program

RETRIEVAL SCORE: 0.4071

TOP RETRIEVED PASSAGES:
1. codenova.txt (score: 0.4071)
CodeNova Internship Program
2. codenova.txt (score: 0.2152)
CodeNova provides hands-on internships in Artificial Intelligence,
Machine Learning, and Web Development.
3. rag_assistant.txt (score: 0.0000)
Retrieval-Augmented Generation, or RAG, retrieves relevant
document passages before generating or constructing an answer.

Enter your question (or type 'exit' to stop): stop
QUESTION: stop

ANSWER:
Information not found in provided document

SOURCE DOCUMENT: None
No supporting source passage was accepted.

TOP RETRIEVED PASSAGES:
1. rag_assistant.txt (score: 0.0000)
Source citations help users verify answers against the documents. If the documents do not support an answer, the assistan

In [9]:
%%writefile app.py

from pathlib import Path
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


DATA_DIR = Path(__file__).parent / "sample_documents"
FALLBACK = "Information not found in provided document"


def load_and_chunk_documents(folder, max_words=100):
    chunks = []

    for file_path in sorted(Path(folder).glob("*.txt")):
        text = file_path.read_text(encoding="utf-8")
        paragraphs = [
            p.strip()
            for p in re.split(r"\n\s*\n", text)
            if p.strip()
        ]

        for paragraph in paragraphs:
            sentences = re.split(r"(?<=[.!?])\s+", paragraph)
            current = []

            for sentence in sentences:
                if not sentence.strip():
                    continue

                proposed = " ".join(current + [sentence])

                if current and len(proposed.split()) > max_words:
                    chunks.append({
                        "source": file_path.name,
                        "text": " ".join(current)
                    })
                    current = [sentence]
                else:
                    current.append(sentence)

            if current:
                chunks.append({
                    "source": file_path.name,
                    "text": " ".join(current)
                })

    return chunks


class LocalQAAssistant:
    def __init__(self, chunks, threshold=0.15, top_k=3):
        if not chunks:
            raise ValueError("No document chunks found.")

        self.chunks = chunks
        self.threshold = threshold
        self.top_k = top_k

        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            sublinear_tf=True
        )

        self.document_vectors = self.vectorizer.fit_transform(
            [item["text"] for item in chunks]
        )

    def answer_query(self, query):
        if not query.strip():
            return {
                "answer": "Please enter a question.",
                "source": None,
                "passage": None,
                "score": 0.0,
                "retrieved": []
            }

        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(
            query_vector, self.document_vectors
        ).flatten()

        indices = np.argsort(scores)[::-1][:self.top_k]

        retrieved = [
            {
                "source": self.chunks[i]["source"],
                "passage": self.chunks[i]["text"],
                "score": float(scores[i])
            }
            for i in indices
        ]

        if not retrieved or retrieved[0]["score"] < self.threshold:
            return {
                "answer": FALLBACK,
                "source": None,
                "passage": None,
                "score": retrieved[0]["score"] if retrieved else 0.0,
                "retrieved": retrieved
            }

        best = retrieved[0]

        return {
            "answer": best["passage"],
            "source": best["source"],
            "passage": best["passage"],
            "score": best["score"],
            "retrieved": retrieved
        }


def main():
    chunks = load_and_chunk_documents(DATA_DIR)
    assistant = LocalQAAssistant(chunks)

    print("Local QA Assistant")
    print("Type 'exit' to quit.")

    while True:
        question = input("\nQuestion: ").strip()

        if question.lower() == "exit":
            break

        result = assistant.answer_query(question)

        print("\nAnswer:", result["answer"])

        if result["passage"]:
            print("Source document:", result["source"])
            print("Supporting passage:", result["passage"])

        print(f"Retrieval score: {result['score']:.4f}")

        if result["retrieved"]:
            print("\nTop retrieved passages:")
            for item in result["retrieved"]:
                print(
                    f"- {item['source']} "
                    f"(score: {item['score']:.4f})"
                )
                print(" ", item["passage"])


if __name__ == "__main__":
    main()

Writing app.py


In [10]:
from google.colab import files
files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import shutil
from google.colab import files

shutil.make_archive(
    "sample_documents",
    "zip",
    root_dir="sample_documents"
)

files.download("sample_documents.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>